In [1]:
import numpy as np
from clarautils import CCol, TableFields, QueryableTable

Define Table Datatype

In [2]:
class DTableFields(TableFields):
    signed: CCol | np.bool_
    abs_min: CCol | np.uint64
    max: CCol | np.uint64
    bits: CCol | np.uint8
    type: CCol | np.object_

Build data to hold

In [3]:
def build_type_tbl():
    kind = {'u': False, 'i': True}
    sizes = [1, 2, 4, 8]
    types = np.array([
        np.dtype(f"{k}{s}")
        for k in kind
        for s in sizes
    ])

    return [(
        kind[t.kind],
        -np.iinfo(t).min,
        np.iinfo(t).max,
        np.iinfo(t).bits,
        t
    ) for t in types]

data = build_type_tbl()
print(data)

[(False, 0, 255, 8, dtype('uint8')), (False, 0, 65535, 16, dtype('uint16')), (False, 0, 4294967295, 32, dtype('uint32')), (False, 0, 18446744073709551615, 64, dtype('uint64')), (True, 128, 127, 8, dtype('int8')), (True, 32768, 32767, 16, dtype('int16')), (True, 2147483648, 2147483647, 32, dtype('int32')), (True, 9223372036854775808, 9223372036854775807, 64, dtype('int64'))]


stick it together

In [4]:
qtbl = QueryableTable(data, DTableFields)
print(qtbl)

QueryableTable(len=8)


The table is typed, so accessing the columns is as easy as it should be, with full pycharm linting support

In [5]:
print(qtbl.signed)

CCol(signed): array([False, False, False, False,  True,  True,  True,  True])


Acces a row

In [6]:
row = qtbl[0]
print(row)

DTable(signed=np.False_, abs_min=np.uint64(0), max=np.uint64(255), bits=np.uint8(8), type=dtype('uint8'))


Row is typed so working with it without nasty workarounds

In [7]:
if row.signed:
    print("Is signed")
else:
    print("Not signed")

Not signed


Actual querying

In [8]:
item = qtbl.type.get_first((qtbl.signed == True) & (qtbl.max > np.iinfo(np.int16).max))
print(item)

int32
